# w4_cpu2_visible_boundary


In [ ]:
import hashlib, io, json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
MOUNT = Path('/kaggle/input')
assert WORK.is_dir(), 'Run on Kaggle'
REV = 'de6ecc4f35be0019d368bb033d35ff113f7e6083'
with tempfile.TemporaryDirectory(prefix='w4_vbf_') as ts:
    tmp = Path(ts)
    repo = tmp / 'repo'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': ''}
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    art_dir, meta_dir, vbf_in, vbf_out = tmp / 'heldout', tmp / 'pinns_meta', tmp / 'vbf_in', tmp / 'vbf_out'
    with zipfile.ZipFile(heldout_bin) as zf:
        zf.extractall(art_dir)
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    vbf_src = repo / 'reviews/pinns_visible_boundary_forecasts'
    vbf_in.mkdir(parents=True, exist_ok=True)
    for fn in ('PROTOCOL.md', 'evaluate_visible_boundaries.py', 'test_visible_boundaries.py'):
        shutil.copy2(vbf_src / fn, vbf_in / fn)
    subprocess.run([sys.executable, str(vbf_in / 'test_visible_boundaries.py')], cwd=vbf_in, env=env, check=True)
    subprocess.run([sys.executable, str(vbf_src / 'prepare_inputs.py'), str(meta_dir), str(art_dir), str(repo / 'reviews/pinns_scene_scope'), str(vbf_in)], cwd=repo, env=env, check=True)
    sealed_tmp = tmp / 'sealed_targets.npz'
    shutil.move(str(vbf_in / 'sealed_targets.npz'), str(sealed_tmp))
    subprocess.run([sys.executable, str(vbf_in / 'evaluate_visible_boundaries.py'), str(vbf_in), str(vbf_out)], cwd=repo, env=env, check=True)
    shutil.move(str(sealed_tmp), str(vbf_in / 'sealed_targets.npz'))
    subprocess.run([sys.executable, str(vbf_in / 'evaluate_visible_boundaries.py'), str(vbf_in), str(vbf_out), '--score'], cwd=repo, env=env, check=True)
    dest = WORK / 'visible_boundary_forecasts.zip'
    shutil.copy2(vbf_out / 'visible_boundary_forecasts.zip', dest)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
